# Query 4: Subqueries and Set Operations

Run `ddl.ipynb` and `dml.ipynb` first. These examples use subqueries to answer questions about products and customers, then combine result sets.

In [ ]:
%load_ext sql
%sql sqlite:///food-analysis/food_analysis.db

## Find products priced above the average

The subquery calculates one value. The outer query compares every product with that average.

In [ ]:
%%sql
SELECT product_name, category, unit_price
FROM products
WHERE unit_price > (SELECT AVG(unit_price) FROM products)
ORDER BY unit_price DESC;

## Find customers with delivered orders

`EXISTS` is true when its subquery finds at least one matching row. The subquery is correlated because it refers to the current customer.

In [ ]:
%%sql
SELECT c.customer_id, c.customer_name
FROM customers AS c
WHERE EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
      AND o.status = 'delivered'
)
ORDER BY c.customer_id
LIMIT 10;

## Combine customer cities and areas

`UNION` removes duplicates; `UNION ALL` keeps them. `INTERSECT` returns values in both results, and `EXCEPT` returns values from the first result that are absent from the second. Each side must return the same number of columns.

In [ ]:
%%sql
SELECT city AS location
FROM customers
WHERE city IS NOT NULL AND TRIM(city) <> ''
UNION
SELECT area AS location
FROM customers
WHERE area IS NOT NULL AND TRIM(area) <> ''
ORDER BY location;

In [ ]:
%%sql
SELECT city AS location FROM customers WHERE city IS NOT NULL
INTERSECT
SELECT area AS location FROM customers WHERE area IS NOT NULL
ORDER BY location;

In [ ]:
%%sql
SELECT city AS location FROM customers WHERE city IS NOT NULL
EXCEPT
SELECT area AS location FROM customers WHERE area IS NOT NULL
ORDER BY location;